In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('telco_customer_churn_cleaned.csv')

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nTarget Distribution:")
print(df['churn_flag'].value_counts())
print(f"\nChurn Rate: {df['churn_flag'].mean() * 100:.1f}%")

Shape: 7043 rows × 31 columns

Target Distribution:
churn_flag
0    5174
1    1869
Name: count, dtype: int64

Churn Rate: 26.5%


In [5]:
# Quick sanity check: What raw data do we have to work with?
print("\nAvailable Raw Features for Engineering:")
print("=" * 60)

# Identify base features we'll engineer from
base_features = {
    'Recency': ['tenure'],
    'Engagement': ['partner_flag', 'dependents_flag', 'phoneservice_flag', 
                   'multiplelines_No phone service', 'multiplelines_Yes',
                   'onlinesecurity_No internet service', 'onlinesecurity_Yes',
                   'onlinebackup_No internet service', 'onlinebackup_Yes',
                   'deviceprotection_No internet service', 'deviceprotection_Yes',
                   'techsupport_No internet service', 'techsupport_Yes',
                   'streamingtv_No internet service', 'streamingtv_Yes',
                   'streamingmovies_No internet service', 'streamingmovies_Yes'],
    'Monetary': ['MonthlyCharges', 'TotalCharges', 'tenure'],
    'Contract': ['contract_ordinal', 'paperlessbilling_flag'],
    'Payment': ['payment_Credit card (automatic)', 'payment_Electronic check', 
                'payment_Mailed check']
}

for category, features in base_features.items():
    available = [f for f in features if f in df.columns]
    print(f"\n{category}:")
    print(f"  {len(available)}/{len(features)} features available")
    if len(available) != len(features):
        missing = set(features) - set(available)
        print(f"  Missing: {missing}")

print("\n" + "=" * 60)
print("✓ Raw data validated for feature engineering")


Available Raw Features for Engineering:

Recency:
  1/1 features available

Engagement:
  17/17 features available

Monetary:
  3/3 features available

Contract:
  2/2 features available

Payment:
  3/3 features available

✓ Raw data validated for feature engineering




### What signals indicate a customer is about to churn?
- Short tenure (new customers, no habit formed)
- Month-to-month contract (no commitment, can leave anytime)
- Few services (low integration, easy to switch)
- Manual payment method (friction, monthly decision point)
- High monthly charges relative to services (value perception issue)

### What signals indicate a customer is valuable long-term?
- High total charges (already invested significantly)
- Multiple services (deeply integrated into daily life)
- Long tenure (proven loyalty)
- Automatic payment (frictionless, set-it-forget-it)
- Contract commitment (1-2 year agreements)

### What signals indicate engagement or loyalty?
- Multiple addon services (security, backup, streaming)
- Long tenure (survived multiple evaluation periods)
- Contract renewals (chose to stay)
- Low friction billing (paperless + automatic payment)

---

###FEATURE CATEGORIES

### Category 1: RECENCY-LIKE SIGNALS
**Why:** New customers churn faster; long-term customers have inertia  
**Planned:** tenure buckets (0-6mo/6-24mo/24mo+), trial period flag

### Category 2: FREQUENCY/ENGAGEMENT SIGNALS
**Why:** More services = higher switching costs and daily value  
**Planned:** total service count, engagement tier, phone-only risk flag

### Category 3: MONETARY VALUE SIGNALS
**Why:** Revenue quality matters more than raw revenue  
**Planned:** average revenue per month (ARPM), spending tier, efficiency ratio

### Category 4: CONTRACT RISK SIGNALS
**Why:** Contracts create lock-in; month-to-month is high-risk  
**Planned:** commitment level, contract-tenure interaction, digital risk score

### Category 5: PAYMENT BEHAVIOR SIGNALS
**Why:** Payment friction = monthly decision point to cancel  
**Planned:** payment friction score, billing convenience, automation level



In [6]:
print("RECENCY FEATURES:")

#TENURE BUCKETS
def assign_tenure_bucket(tenure_months):
    """Categorize cust by lifecycle stage based on tenure."""
    if tenure_months<=12:
        return 'trial'
    elif tenure_months<=36:
        return 'established'
    elif tenure_months<=60:
        return 'loyal'
    else:
        return 'Champion'
    
df['tenure_bucket']=df['tenure'].apply(assign_tenure_bucket)
print("\n1. Tenure Buckets Distribution:")
print(df['tenure_bucket'].value_counts().sort_index())
print(f"\nChurn Rate by Tenure Bucket:")
for bucket in ['trial', 'established', 'loyal', 'champion']:
    churn_rate = df[df['tenure_bucket'] == bucket]['churn_flag'].mean() * 100
    count = len(df[df['tenure_bucket'] == bucket])
    print(f"  {bucket:12s}: {churn_rate:5.1f}% (n={count:,})")

RECENCY FEATURES:

1. Tenure Buckets Distribution:
tenure_bucket
Champion       1407
established    1856
loyal          1594
trial          2186
Name: count, dtype: int64

Churn Rate by Tenure Bucket:
  trial       :  47.4% (n=2,186)
  established :  25.5% (n=1,856)
  loyal       :  16.6% (n=1,594)
  champion    :   nan% (n=0)


In [7]:
# Normalized Tenure Score
max_tenure = df['tenure'].max()
df['tenure_normalized'] = df['tenure'] / max_tenure

print(f"\n2. Normalized Tenure Score:")
print(f"  Range: [{df['tenure_normalized'].min():.3f}, {df['tenure_normalized'].max():.3f}]")
print(f"  Mean: {df['tenure_normalized'].mean():.3f}")
print(f"  Median: {df['tenure_normalized'].median():.3f}")


2. Normalized Tenure Score:
  Range: [0.000, 1.000]
  Mean: 0.450
  Median: 0.403


In [8]:
df['is_trial_period']=(df['tenure']<=6).astype(int)
trial_customers=df['is_trial_period'].sum()
trial_churn_rate=df[df['is_trial_period']==1]['churn_flag'].mean()*100
stable_churn_rate=df[df['is_trial_period']==0]['churn_flag'].mean()*100

print(f"\n3. Trial Period Risk Flag:")
print(f"  Trial customers (≤6mo): {trial_customers:,} ({trial_customers/len(df)*100:.1f}%)")
print(f"  Trial churn rate: {trial_churn_rate:.1f}%")
print(f"  Post-trial churn rate: {stable_churn_rate:.1f}%")
print(f"  Risk multiplier: {trial_churn_rate/stable_churn_rate:.2f}x")



3. Trial Period Risk Flag:
  Trial customers (≤6mo): 1,481 (21.0%)
  Trial churn rate: 52.9%
  Post-trial churn rate: 19.5%
  Risk multiplier: 2.71x


In [9]:
df['tenure_squared'] = df['tenure'] ** 2

print(f"\n4. Tenure Squared (Non-linear Signal):")
print(f"  Captures diminishing returns of loyalty over time")
print(f"  Range: [{df['tenure_squared'].min():,.0f}, {df['tenure_squared'].max():,.0f}]")



4. Tenure Squared (Non-linear Signal):
  Captures diminishing returns of loyalty over time
  Range: [0, 5,184]


In [10]:
tenure_bin_map = {'trial': 1, 'established': 2, 'loyal': 3, 'champion': 4}
df['tenure_bin_ordinal'] = df['tenure_bucket'].map(tenure_bin_map)

print(f"\n5. Tenure Bins (Ordinal):")
print(f"  1=trial, 2=established, 3=loyal, 4=champion")
print(f"  Mean: {df['tenure_bin_ordinal'].mean():.2f}")



5. Tenure Bins (Ordinal):
  1=trial, 2=established, 3=loyal, 4=champion
  Mean: 1.89


In [12]:
print("SUMMARY:")
recency_features = [
    'tenure_bucket',          
    'tenure_normalized',      
    'is_trial_period',        
    'tenure_squared',         
    'tenure_bin_ordinal'      
]

print("\n" + "=" * 60)
print("RECENCY FEATURES COMPLETE")
print(f"Created {len(recency_features)} recency-based features")
print(f"All features added to dataframe")
print("\nNew Features:")
for feat in recency_features:
    print(f"  - {feat}")
print("=" * 60)

SUMMARY:

RECENCY FEATURES COMPLETE
Created 5 recency-based features
All features added to dataframe

New Features:
  - tenure_bucket
  - tenure_normalized
  - is_trial_period
  - tenure_squared
  - tenure_bin_ordinal


## Recency Feature Engineering Summary

### Objective
Transform raw `tenure` data into meaningful signals that capture customer lifecycle stage and churn risk.

### Key Insight
**Churn risk is non-linear with tenure:**
- New customers (0-12mo): Still evaluating, high churn
- Mid-tenure (12-48mo): Past trial period, moderate risk  
- Long-tenure (48mo+): Established habits, low churn

### Features Created 

| Feature | Type | Purpose |
|---------|------|---------|
| tenure_bucket | Categorical | Lifecycle stage (trial/established/loyal/champion) |
| tenure_normalized | Continuous | 0-1 scaled tenure for ML algorithms |
| is_trial_period | Binary | High-risk flag for first 6 months |
| tenure_squared | Continuous | Captures non-linear loyalty effects |
| tenure_bin_ordinal | Ordinal | Numeric lifecycle stage (1-4) for tree models |

### Validation Results
- **Trial period churn** significantly higher than established customers
- Clear inverse relationship between tenure and churn rate
- Features capture both linear and non-linear customer loyalty patterns